# Embedding Articles for ML-Ontology

In [1]:
# Imports

import pandas as pd
from pathlib import Path
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
from collections import Counter

In [11]:
# Paths

processed_abstracts_path = Path("../literature-mining/data/processed/abstracts")
save_embeddings_path = Path("embeddings")

# Ensure directories exist
for p in [save_embeddings_path]:
    p.mkdir(parents=True, exist_ok=True)

print("All directories verified/created.")



All directories verified/created.


In [6]:
# Load dataset
df_abstracts = pd.read_csv(processed_abstracts_path / "abstracts.csv")
print(f"Loaded {len(df_abstracts)} abstracts.")

Loaded 52290 abstracts.


In [7]:
# Remove abstract duplicates

# Count how many rows each query_id has
query_counts = df_abstracts["query_id"].value_counts().to_dict()

# Create a copy and map the counts to each row
df_abstracts = df_abstracts.copy()
df_abstracts["query_size"] = df_abstracts["query_id"].map(query_counts)

# Sort so that query groups with fewer rows are prioritized
df_abstracts_sorted = df_abstracts.sort_values(by="query_size", ascending=True)

# Remove duplicate DOIs, keeping the one in the smallest query group
df_abstracts_dedup = df_abstracts_sorted.drop_duplicates(subset="doi", keep="first").drop(columns=["query_size"])

# Print results
print("Original dataset size:", len(df_abstracts))
print("After removing duplicates:", len(df_abstracts_dedup))
print("Remaining duplicate DOIs:", df_abstracts_dedup["doi"].duplicated().sum())

df_abstracts = df_abstracts_dedup

Original dataset size: 52290
After removing duplicates: 33130
Remaining duplicate DOIs: 0


In [8]:
# Load embedding model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [9]:
tokenizer = model.tokenizer  # tokenizer associated with MPNet model
max_len = model.get_max_seq_length()  # typically 384 for MPNet

def embed_with_chunking(text):
    tokens = tokenizer.tokenize(text)

    # If below limit -> embed normally
    if len(tokens) <= max_len:
        return model.encode(
            text,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

    # Split into chunks that fit model's input size
    chunks = [
        tokenizer.convert_tokens_to_string(tokens[i:i+max_len])
        for i in range(0, len(tokens), max_len)
    ]

    # Embed each chunk and average
    chunk_embeddings = model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.mean(chunk_embeddings, axis=0)




## Embedding of Abstracts (no need to run this multiple times)

In [10]:
# Batch embedding loop with chunking 

texts = df_abstracts["clean_abs"].astype(str).tolist()
batch_size = 512

embeddings = []

for start in tqdm(range(0, len(texts), batch_size), desc="Embedding abstracts"):
    batch = texts[start:start+batch_size]

    batch_emb = [embed_with_chunking(text) for text in batch]
    embeddings.append(np.vstack(batch_emb))

embeddings = np.vstack(embeddings)

print("Embedding matrix shape:", embeddings.shape)


Embedding abstracts: 100%|██████████| 65/65 [07:18<00:00,  6.75s/it]

Embedding matrix shape: (33130, 768)


In [15]:
# Save embeddings to .npy file
emb_path = save_embeddings_path / "abstract_embeddings.npy"
np.save(emb_path, embeddings)
print("Saved embeddings to:", emb_path)

Saved embeddings to: embeddings/abstract_embeddings.npy


In [12]:
# DOI list in embedding row order
doi_ids = df_abstracts_dedup["doi"].astype(str).to_numpy()

# Sanity check
assert len(doi_ids) == embeddings.shape[0]

# Save IDs and a direct mapping table
np.save(save_embeddings_path / "abstract_embedding_dois.npy", doi_ids)

mapping_df = pd.DataFrame({
    "doi": doi_ids,
    "embedding_row": np.arange(len(doi_ids))
})
mapping_df.to_csv(save_embeddings_path / "abstract_embedding_mapping.csv", index=False)


## Embedding of Titles (no need to run this multiple times)

In [ ]:
# Embed titles
title_texts = df_abstracts["title"].astype(str).tolist()
batch_size = 512

title_embeddings = []
for start in tqdm(range(0, len(title_texts), batch_size), desc="Embedding titles"):
    batch = title_texts[start:start+batch_size]
    batch_emb = model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    title_embeddings.append(batch_emb)
    
title_embeddings = np.vstack(title_embeddings)


Embedding titles: 100%|██████████| 65/65 [00:15<00:00,  4.32it/s]


In [16]:
# Save embeddings to .npy file
emb_path = save_embeddings_path / "title_embeddings.npy"
np.save(emb_path, title_embeddings)
print("Saved title embeddings to:", emb_path)

Saved title embeddings to: embeddings/title_embeddings.npy


In [ ]:
# DOI list in title embedding row order
title_doi_ids = df_abstracts_dedup["doi"].astype(str).to_numpy()

# Sanity check
assert len(title_doi_ids) == title_embeddings.shape[0]

# Save IDs and a direct mapping table for title embeddings
np.save(save_embeddings_path / "title_embedding_dois.npy", title_doi_ids)

title_mapping_df = pd.DataFrame({
    "doi": title_doi_ids,
    "embedding_row": np.arange(len(title_doi_ids))
})
title_mapping_df.to_csv(save_embeddings_path / "title_embedding_mapping.csv", index=False)
